# Домашнее задание: Pydantic


## Важно!

- При выполнении задания используем точные типы (`EmailStr`, `HttpUrl`, `SecretStr`, `Decimal`, конкретные `Enum`).
- Придерживаемся принципа разделения валидаций: проверка поля — в `field_validator`, сквозные зависимости — в `model_validator`


## Задача 1. Профиль пользователя (валидация полей)

Постройте модель профиля пользователя для внутренней CRM:

**Требования**
1. Обязательные поля: `id: UUID`, `email: EmailStr`, `name: str`.
2. Опциональные поля: `website: HttpUrl | None`, `bio: str | None`.
3. Пароль хранится как `SecretStr`, должен быть не короче 8 символов.
4. Имя (`name`) нормализуйте: тримминг + одна пробельная последовательность между словами + первая буква каждого слова заглавная.
5. Если указан `website`, домен сайта не должен совпадать с доменом `email` (смысл: личный сайт != корпоративная почта).

Подсказки: используйте `field_validator` для нормализации и локальных проверок; и `model_validator(mode="after")` для проверки зависимости `email` ↔ `website`.


In [1]:
!pip install -U pydantic[email,timezone] -q

In [2]:
from typing import Optional
from pydantic import BaseModel, Field, EmailStr, HttpUrl, SecretStr
from pydantic import field_validator, model_validator
from uuid import UUID

class UserProfile(BaseModel):
    # Обязательные поля
    id: UUID
    email: EmailStr
    name: str
    password: SecretStr

    # Опциональные поля
    website: Optional[HttpUrl] = None
    bio: Optional[str] = None

    # Нормализация имени:
    # - trim
    # - одна пробельная последовательность между словами
    # - каждое слово с заглавной буквы
    @field_validator("name")
    @classmethod
    def normalize_name(cls, v: str) -> str:
        v = v.strip()
        if not v:
            raise ValueError("Имя не может быть пустым")

        # Одна последовательность пробелов между словами
        v = " ".join(v.split())

        # Каждое слово с заглавной буквы
        return v.title()

    # Проверка длины пароля (не короче 8 символов)
    @field_validator("password")
    @classmethod
    def password_strength(cls, v: SecretStr) -> SecretStr:
        raw = v.get_secret_value()
        if len(raw) < 8:
            raise ValueError("Пароль должен быть не короче 8 символов")
        return v

    # Сквозная проверка: домены email и website не должны совпадать
    @model_validator(mode="after")
    def check_domains(self):
        if self.website is None or self.email is None:
            return self

        # домен из email
        email_domain = self.email.split("@", 1)[1].lower()

        # домен сайта
        website_domain = self.website.host.lower()
        # считаем www.example.com тем же доменом, что и example.com
        if website_domain.startswith("www."):
            website_domain = website_domain[4:]

        if email_domain == website_domain:
            raise ValueError(
                "Домен личного сайта не должен совпадать с доменом корпоративной почты"
            )

        return self

## Задача 2. Валидация функции заказа (`@validate_call`)

Реализуйте функцию `place_order`, которая принимает:
- `user_id: UUID`
- `sku: str` (артикул, только заглавные буквы/цифры, длина 3–12)
- `quantity: int` (>0)
- `price: Decimal` (>= 0), округляется банковским методом до 2 знаков

Функция должна возвращать словарь с ключами: `user_id`, `sku`, `quantity`, `price`, `amount` (quantity × price).

Используйте `@validate_call` и локальные проверки через обычный код (или вспомогательные валидаторы `TypeAdapter` не используем).


In [3]:
from pydantic import validate_call
from decimal import Decimal, ROUND_HALF_EVEN
from uuid import UUID
import re

# TODO: реализуйте функцию с @validate_call
@validate_call
def place_order(
    user_id: UUID,
    sku: str,
    quantity: int,
    price: Decimal,
) -> dict:
    # Проверка SKU: только A-Z и 0-9, длина 3–12
    if not re.fullmatch(r"[A-Z0-9]{3,12}", sku):
        raise ValueError("SKU должен содержать только заглавные буквы и цифры, длина 3–12 символов")

    # Проверка количества
    if quantity <= 0:
        raise ValueError("Количество должно быть положительным целым числом")

    # Проверка и нормализация цены
    if price < 0:
        raise ValueError("Цена не может быть отрицательной")

    # Округление банковским методом до 2 знаков
    price = price.quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN)

    amount = price * quantity

    return {
        "user_id": user_id,
        "sku": sku,
        "quantity": quantity,
        "price": price,
        "amount": amount,
    }

## Задача 3. Модель заказа с бизнес-правилами

Смоделируйте заказ в магазине цифровых товаров.

**Требования**
- `OrderStatus: Enum` со значениями `new`, `paid`, `delivered`, `canceled`.
- Модель `OrderItem`:
  - `sku: str` как в задаче 2
  - `qty: int` (>0)
  - `unit_price: Decimal` (>=0) округление до 2 знаков
- Модель `Order`:
  - `id: UUID`
  - `user_email: EmailStr`
  - `items: list[OrderItem]` (не пустой)
  - `status: OrderStatus = 'new'`
  - `created_at: datetime` (по умолчанию `datetime.utcnow`)
  - Расчитанное поле `total: Decimal` — сумма по всем позициям
  - В `model_validator(mode="after")` запретите переход в `paid`/`delivered` при `total == 0` и запретите пустые корзины.

**Важно:** используйте только инструменты `pydantic` и стандартную библиотеку.


In [4]:
from pydantic import BaseModel, EmailStr, field_validator, model_validator, Field, computed_field
from typing import List
from decimal import Decimal, ROUND_HALF_EVEN
from uuid import UUID
from datetime import datetime
from enum import Enum
import re

SKU_RE = re.compile(r"^[A-Z0-9]{3,12}$")

class OrderStatus(str, Enum):
    new = "new"
    paid = "paid"
    delivered = "delivered"
    canceled = "canceled"


class OrderItem(BaseModel):
    sku: str
    qty: int
    unit_price: Decimal

    @field_validator("sku")
    @classmethod
    def sku_format(cls, v: str) -> str:
        if not SKU_RE.fullmatch(v):
            raise ValueError("SKU должен содержать 3–12 символов: только A-Z и 0-9")
        return v

    @field_validator("qty")
    @classmethod
    def qty_positive(cls, v: int) -> int:
        if v <= 0:
            raise ValueError("Количество (qty) должно быть > 0")
        return v

    @field_validator("unit_price")
    @classmethod
    def price_non_negative_and_round(cls, v: Decimal) -> Decimal:
        if v < 0:
            raise ValueError("Цена не может быть отрицательной")
        # банковское округление до 2 знаков
        return v.quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN)


class Order(BaseModel):
    id: UUID
    user_email: EmailStr
    items: List[OrderItem]
    status: OrderStatus = OrderStatus.new
    created_at: datetime = Field(default_factory=datetime.utcnow)

    @computed_field
    @property
    def total(self) -> Decimal:
        total = Decimal("0.00")
        for item in self.items:
            total += item.unit_price * item.qty
        # приведем к 2 знакам на всякий случай
        return total.quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN)

    @model_validator(mode="after")
    def check_business_rules(self):
        # запрет пустой корзины
        if not self.items:
            raise ValueError("Заказ не может быть без позиций (пустая корзина)")

        # запрет оплаченных/доставленных заказов с total == 0
        if self.status in {OrderStatus.paid, OrderStatus.delivered} and self.total == Decimal(
            "0.00"
        ):
            raise ValueError(
                "Нельзя перевести заказ в статус paid/delivered при нулевой сумме"
            )

        return self

## Задача 4. Конфигурация приложения (`BaseSettings`)

Опишите настройки подключения к внешнему API:

- `APISettings(BaseSettings)` с полями:
  - `base_url: HttpUrl`
  - `token: SecretStr`
  - `timeout_sec: int = 5` (1–60)
  - `retries: int = 2` (0–10)
- Используйте `model_config = ConfigDict(env_prefix="API_", env_file=".env", extra="ignore")`
- Проверьте, что значения корректно читаются из переменных окружения.

В тесте ниже среда заполняется вручную.


In [6]:
!pip install -U pydantic_settings -q

In [7]:
import os
from pydantic_settings import BaseSettings
from pydantic import ConfigDict, SecretStr, HttpUrl, field_validator
from pydantic import FieldValidationInfo  # для info.field_name


class APISettings(BaseSettings):
    base_url: HttpUrl
    token: SecretStr
    timeout_sec: int = 5   # 1–60
    retries: int = 2       # 0–10

    # пример проверки диапазона для timeout_sec / retries
    @field_validator("timeout_sec", "retries")
    @classmethod
    def check_ranges(cls, v: int, info: FieldValidationInfo):
        if info.field_name == "timeout_sec":
            if not (1 <= v <= 60):
                raise ValueError("timeout_sec должен быть в диапазоне 1–60 секунд")
        elif info.field_name == "retries":
            if not (0 <= v <= 10):
                raise ValueError("retries должен быть в диапазоне 0–10")
        return v

    model_config = ConfigDict(
        env_prefix="API_",   # имена переменных окружения начинаются с API_
        env_file=".env",     # можно дополнительно читать из .env
        extra="ignore",      # игнорировать лишние поля
    )

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_20192\3495511909.py:4: PydanticDeprecatedSince20: Importing FieldValidationInfo from `pydantic` is deprecated. This feature is either no longer supported, or is not public. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  from pydantic import FieldValidationInfo  # для info.field_name


## Задача 5. Извлечение из ORM (`from_attributes=True`)

Создайте простую SQLAlchemy-модель `SAUser(id, email, is_active)` (in-memory, без БД) и соответствующую модель Pydantic:

- Pydantic-модель `UserOut` с полями `id: UUID`, `email: EmailStr`, `is_active: bool`.
- Включите поддержку `from_attributes` в `model_config`.
- Создайте инстанс `SAUser` и провалидируйте его через `UserOut.model_validate(sa_user_instance)`.

Проверьте, что преобразование сработало.


In [8]:
!pip install -U sqlalchemy -q

In [9]:
from typing import Optional
from sqlalchemy import Column, String, Boolean
from sqlalchemy.orm import declarative_base
from uuid import uuid4
from pydantic import BaseModel, EmailStr, ConfigDict

Base = declarative_base()

class SAUser(Base):
    __tablename__ = "users"
    id = Column(String, primary_key=True, default=lambda: str(uuid4()))
    email = Column(String, nullable=False)
    is_active = Column(Boolean, default=True)

    def __init__(self, email: str, is_active: bool = True):
        self.id = str(uuid4())
        self.email = email
        self.is_active = is_active

class UserOut(BaseModel):
    id: UUID
    email: EmailStr
    is_active: bool

    model_config = ConfigDict(
        from_attributes=True  # важное место: позволяет валидировать из ORM-объекта
    )


# ----- Проверка преобразования -----

# создаём ORM-объект (без фактического коннекта к БД)
sa_user = SAUser(email="user@example.com", is_active=True)

# валидируем его через Pydantic
user_out = UserOut.model_validate(sa_user)

print(user_out)
print(user_out.model_dump())
# UserOut id=UUID('...') email='user@example.com' is_active=True
# {'id': UUID('...'), 'email': 'user@example.com', 'is_active': True}

id=UUID('f2c7facd-5ef2-472b-8051-3bc8bdeeefd9') email='user@example.com' is_active=True
{'id': UUID('f2c7facd-5ef2-472b-8051-3bc8bdeeefd9'), 'email': 'user@example.com', 'is_active': True}


## Задача 6. JSON Schema и дружелюбные ошибки

1. Для модели из задачи 3 сгенерируйте JSON Schema (метод `model_json_schema`) и запишите его в переменную `ORDER_SCHEMA`.
2. Реализуйте функцию `safe_create_order(data: dict) -> tuple[bool, str]`, которая:
   - пытается создать `Order` из входного `dict`,
   - при успехе возвращает `(True, "<total=...>")`,
   - при ошибке возвращает `(False, "<короткое сообщение об ошибке>")` без стек-трейса.

Не используйте сторонние библиотеки.


In [10]:
# Используем модели из задачи 3: OrderStatus, OrderItem, Order
from pydantic import ValidationError

ORDER_SCHEMA = Order.model_json_schema()  # TODO: сгенерируйте схему

def safe_create_order(data: dict) -> tuple[bool, str]:
    try:
        order = Order(**data)
        # успех — возвращаем total
        return True, f"<total={order.total}>"
    except ValidationError as e:
        # берём первую ошибку и короткое сообщение
        first = e.errors()[0] if e.errors() else {}
        msg = first.get("msg", "Invalid data")
        return False, f"<{msg}>"
    except Exception as e:
        # на всякий случай перехватываем прочие ошибки
        return False, f"<{str(e)}>"